# bgm_ace_step

**ACE-Step 1.5** 로 BGM 생성. Stable Audio 3와 비교하기 위한 대조군.

| | |
|---|---|
| 커널 | `ace` |
| 출력 | `outputs/bgm_ace_step/<시나리오>/` |
| 개수 | 시나리오당 페이즈 2 x 후보 3 = 6개 |

태그 기반 제어가 본래 인터페이스라 산문이 아니라 스타일 태그를 넣는다.

## 설치가 까다롭다

- **파이썬 3.11 이상** 필요 (3.10에서는 설치 자체가 거부된다)
- `nano-vllm` 이 PyPI에 없어 **GitHub에서 먼저 설치**해야 한다

아래 1번 셀이 그 순서를 지킨다. 그래도 실패하면 이 노트북은 건너뛰어도 된다 —
`bgm_stable_audio` 만으로 "문화 조건이 반영되는가"라는 질문에는 답이 나온다.


## 1. 커널 확인

설치는 `setup_kernels.ipynb` 가 한다 (`uv sync`).

`pip install git+...` 로는 설치가 **불가능**하다. pyproject 가 linux x86_64 에서
`torch==2.10.0+cu128` 을 요구하는데 그 버전은 PyPI 가 아니라 `[tool.uv.index]` 로
선언된 cu128 인덱스에만 있고, `nano-vllm` 도 `[tool.uv.sources]` 매핑이다.
pip 은 그 설정을 읽지 않는다.

In [ ]:
# ---------------------------------------------------------------------------
# 커널 확인. 설치는 setup_kernels.ipynb 가 담당한다.
#
# 이 셀에 pip install 을 추가하지 말 것. 노트북 4개는 서로 양립 불가능한
# torch/transformers 를 쓰기 때문에, 여기서 다시 깔면 커널 조합이 깨진다.
#   ace  torch 2.10.0+cu128 / transformers >=4.51,<4.58
#   sa3  torch 2.7.1 cu126  / transformers >=5.8
#   flux torch 2.8.0        / transformers ==4.56.1
# ---------------------------------------------------------------------------
import sys, importlib

KERNEL = "ace"
NEED   = ["torch", "soundfile", "acestep"]

print("python     :", sys.version.split()[0])
print("interpreter:", sys.executable)
print()

missing = []
for m in NEED:
    try:
        mod = importlib.import_module(m)
        print(f"OK   {m:16s} {getattr(mod, '__version__', '')}")
    except Exception as e:
        missing.append(m)
        print(f"FAIL {m:16s} {type(e).__name__}: {e}")

assert not missing, (
    f"\n{missing} 를 import 하지 못했다.\n"
    f"커널을 '{KERNEL}' 로 바꿀 것 (Kernel > Change kernel).\n"
    f"'{KERNEL}' 커널이 목록에 없으면 setup_kernels.ipynb 를 먼저 실행한다.\n"
    f"여기서 pip install 로 때우지 말 것 — 버전 조합이 깨진다."
)
print("\n커널 OK")

## 2. 설정


In [ ]:
import os, sys, json, pathlib, getpass, shutil

USER = getpass.getuser()

# ---- 이 노드에서 유효한 루트를 직접 찾는다 --------------------------------
# 학과 클러스터는 같은 저장소가 노드마다 다른 경로에 마운트된다.
#   로그인 노드(hpcmaster) : /data1/$USER   (= HOME)
#   계산 노드(hpc-stat1)   : /mnt/data1/$USER
# setup_kernels 로 만든 커널이면 PROJ_BASE 가 kernel.json 에 박혀 있고,
# Slurm 으로 돌리면 common.sh 가 환경변수로 넘겨준다. 둘 다 아니면 여기서 찾는다.
def pick_root():
    for c in (f"/mnt/data1/{USER}", f"/data1/{USER}",
              str(pathlib.Path.home()), "/workspace", "/scratch"):
        if pathlib.Path(c).is_dir() and os.access(c, os.W_OK):
            return c
    return None

ROOT = os.environ.get("PROJ_ROOT") or pick_root()
if ROOT is None:
    raise SystemExit(f"쓸 수 있는 루트를 못 찾았다 (user={USER})")

# 계산 노드에는 $HOME 이 아예 없다. 두면 캐시/kernelspec 이 전부 깨진다.
if not pathlib.Path(os.environ.get("HOME", "/nonexistent")).is_dir():
    os.environ["HOME"] = ROOT

BASE  = pathlib.Path(os.environ.get("PROJ_BASE") or f"{ROOT}/mystery")
MODEL = "bgm_ace_step"
OUT_DIR = BASE / "outputs" / MODEL

# 캐시는 홈이 아니라 대용량 스토리지로. 홈 쿼터면 다운로드 중 os error 28 이 난다.
os.environ.setdefault("HF_HOME",      str(BASE / "hf_cache"))
os.environ.setdefault("TMPDIR",       str(BASE / "tmp"))
os.environ.setdefault("MPLCONFIGDIR", str(BASE / "mpl_cache"))

try:
    for p in (OUT_DIR, pathlib.Path(os.environ["HF_HOME"]),
              pathlib.Path(os.environ["TMPDIR"])):
        p.mkdir(parents=True, exist_ok=True)
except (PermissionError, OSError) as e:
    raise SystemExit(
        f"{BASE} 에 쓸 수 없다 -> {type(e).__name__}: {e}\n"
        f"  이 셀 맨 위에서 지정할 것:\n"
        f"      os.environ['PROJ_BASE'] = '/data1/{USER}/mystery'"
    )

# ---- 입력 JSON (scenarios_en/*.json 전부) ---------------------------------
# papermill 로 돌리면 cwd 가 /tmp 라 "." 만으로는 못 찾는다.
# PROJ_INPUT(환경변수) 을 먼저 보고, 없으면 후보들을 뒤진다.
def pick_input_dir():
    cands = []
    if os.environ.get("PROJ_INPUT"):
        cands.append(pathlib.Path(os.environ["PROJ_INPUT"]))
    cands += [pathlib.Path.cwd(), pathlib.Path.cwd().parent, BASE]
    for c in cands:
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    # 마지막 수단: ROOT 아래를 얕게 훑는다
    for c in pathlib.Path(ROOT).glob("*/*/"):
        if (c / "scenarios_en").is_dir() and list((c / "scenarios_en").glob("*.json")):
            return c
    return None

IN_DIR = pick_input_dir()
if IN_DIR is None:
    raise SystemExit(
        "scenarios_en/*.json 을 못 찾았다.\n"
        f"  PROJ_INPUT = {os.environ.get('PROJ_INPUT', '(미설정)')}\n"
        f"  cwd        = {pathlib.Path.cwd()}\n"
        f"  이 셀 맨 위에서:  os.environ['PROJ_INPUT'] = '/data1/{USER}/<저장소>/Jaemin'"
    )

# scenario_id(파일 내부 필드, 예: "12_heungbu")를 키로 쓴다 — 파일명과 무관하게 안정적.
SCENARIO_FILES = sorted((IN_DIR / "scenarios_en").glob("*.json"))
assert SCENARIO_FILES, f"scenarios_en 에 json 이 없다: {IN_DIR / 'scenarios_en'}"

DATA = {}
for _f in SCENARIO_FILES:
    _d = json.load(open(_f, encoding="utf-8"))
    DATA[_d["scenario_id"]] = _d

for k in DATA:
    (OUT_DIR / k).mkdir(parents=True, exist_ok=True)


def out_path(scen, *parts):
    """OUT_DIR/<시나리오>/<...> 경로를 만들고 돌려준다."""
    p = OUT_DIR / scen
    for x in parts[:-1]:
        p = p / x
    p.mkdir(parents=True, exist_ok=True)
    return p / parts[-1]


print("node    :", os.uname().nodename)
print("ROOT    :", ROOT)
print("BASE    :", BASE, f"(여유 {shutil.disk_usage(BASE).free/1e9:.0f} GB)")
print("IN_DIR  :", IN_DIR)
print("HF_HOME :", os.environ["HF_HOME"])
print("OUT_DIR :", OUT_DIR)
print(f"시나리오 {len(DATA)}개:")
for k, d in DATA.items():
    print(f"  {k:24s} 장소 {len(d['places'])} / 인물 {len(d['cast'])}")


In [ ]:
import gc, torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print("torch  :", torch.__version__)
print("device :", DEVICE)
if DEVICE == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("GPU    :", p.name, "|", round(p.total_memory / 1e9, 1), "GB",
          "| compute capability", f"{p.major}.{p.minor}")
    if (p.major, p.minor) < (8, 9):
        print("  주의: FP8 연산 미지원 세대. FP8 가중치를 요구하는 모델은 못 돌린다.")
else:
    print("  GPU가 없다. 로그인 노드이거나 GPU 할당이 안 된 상태.")


def free():
    """모델을 내린 뒤 호출. 커널을 종료하는 것이 가장 확실하다."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    if torch.cuda.is_available():
        used = torch.cuda.memory_allocated() / 1e9
        print(f"할당된 VRAM: {used:.2f} GB")


## 4. 모델 초기화

In [ ]:
# ---------------------------------------------------------------------------
# ACE-Step 1.5 생성
#
# `acestep.pipeline_ace_step` / `ACEStepPipeline` 은 1.5 에 없다 (1.0 경로다).
# 아래는 공식 cli.py 의 초기화 순서를 그대로 옮긴 것이다:
#   AceStepHandler() -> get_available_acestep_v15_models() -> initialize_service()
#   LLMHandler()     -> thinking=False 면 초기화하지 않는다 (cli.py 도 동일)
#   generate_music(dit, llm, params, config, save_dir=...)
#
# 반환값은 오디오 배열이 아니라 '저장된 파일 경로'다. 그래서 sf.write 를 쓰지 않는다.
# 모델 출력은 48kHz 이고, 44100 으로 직접 쓰면 에러 없이 8.8% 느려지고 음이 낮아진다.
# ---------------------------------------------------------------------------
import pathlib
import acestep
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.inference import GenerationParams, GenerationConfig, generate_music

ACE_DIR = pathlib.Path(acestep.__file__).resolve().parent.parent
print("ACE-Step 저장소:", ACE_DIR)

dit = AceStepHandler()
llm = LLMHandler()          # thinking=False 라 초기화하지 않는다

models = dit.get_available_acestep_v15_models()
if not models:
    print("체크포인트가 없다. 내려받는다 (오래 걸린다).")
    from acestep.model_downloader import ensure_main_model, get_checkpoints_dir
    ok, msg = ensure_main_model(get_checkpoints_dir())
    print(msg)
    models = dit.get_available_acestep_v15_models()

assert models, "DiT 모델을 찾지 못했다. acestep 체크포인트 다운로드를 확인할 것."
print("사용 가능한 모델:", models)

# turbo 는 CFG 를 지원하지 않는다. base 가 있으면 그쪽을 쓴다.
MODEL_CFG = next((m for m in models if "turbo" not in m.lower()), models[0])
IS_TURBO  = "turbo" in MODEL_CFG.lower()
print("선택:", MODEL_CFG, "(turbo — guidance_scale 무시됨)" if IS_TURBO else "")

dit.initialize_service(
    project_root=str(ACE_DIR),
    config_path=MODEL_CFG,
    device=DEVICE,
    use_flash_attention=dit.is_flash_attention_available(DEVICE),
    compile_model=False,
    offload_to_cpu=False,
    offload_dit_to_cpu=False,
)
print("핸들러 준비됨")

## 5. 큐 조립

In [ ]:
# ---------------------------------------------------------------------------
# 확장 BGM 큐 파일럿 — 12_흥부전 하나만.
# cues(7) + place_cues(6장소x2상태=12) + character_themes(5) + event_cues(12)
# = 36개.
#
# json/ 의 bgm.*.prompt 는 한국어 음악 용어가 섞인 작곡 참고 문서라(예:
# "상행 4도 도약 뒤 순차 하행"), 그대로 caption 에 넣으면 ACE-Step 이 절반은
# 못 알아듣는 텍스트를 받는 셈이다. 실제로 place_cues/character_themes 가
# 장소·인물에 관계없이 거의 똑같이 들린다는 피드백이 있었다 — bpm/mood 등
# 구조화된 필드는 장소마다 다른데, 그걸 못 읽고 뭉뚱그려 비슷하게 나온 것으로
# 보인다. bgm_stable_audio_expanded.ipynb 에서 쓴 것과 같은 번역 데이터로,
# story/case 에 이미 쓰고 있는 ace_tags() 스타일(쉼표로 나열된 짧은 태그,
# 문장이 아님)에 맞춰 build_tags() 를 새로 만든다. "cinematic" 같은 서양
# 영화음악 쪽으로 끌리는 태그는 넣지 않는다.
# ---------------------------------------------------------------------------
import re, json as _json

PILOT_ID = "12_heungbu"
PILOT_JSON = IN_DIR / "json" / "12_흥부전.json"
assert PILOT_JSON.exists(), f"파일럿 원본을 못 찾음: {PILOT_JSON}"
_raw = _json.load(open(PILOT_JSON, encoding="utf-8"))
bgm_full = _raw["bgm"]


def parse_seconds(form_text, default=60):
    m = re.search(r"(\d+)\s*[~-]\s*(\d+)\s*초", form_text or "")
    if m:
        return (int(m.group(1)) + int(m.group(2))) / 2
    m = re.search(r"(\d+)\s*초", form_text or "")
    return int(m.group(1)) if m else default


jobs = []
for cid, c in bgm_full["cues"].items():
    jobs.append(("cues", cid, c))
for pid, p in bgm_full["place_cues"].items():
    for state in ("pre_murder", "post_murder"):
        jobs.append(("place_cues", f"{pid}_{state}", p[state]))
for cid, c in bgm_full["character_themes"].items():
    jobs.append(("character_themes", cid, c))
for eid, e in bgm_full["event_cues"].items():
    jobs.append(("event_cues", eid, e))

print(f"{PILOT_ID} 확장 BGM: {len(jobs)}개 큐")
for cat, cid, c in jobs:
    secs = parse_seconds(c.get("form"))
    tag = "one-shot" if "one-shot" in (c.get("form") or "") else "loop"
    print(f"  [{cat}/{cid}] {secs:.0f}s ({tag})")


INSTR_EN = {
    "대금": "daegeum bamboo flute", "해금": "haegeum two-string fiddle",
    "거문고": "geomungo six-string zither", "아쟁": "ajaeng bowed zither",
    "장구": "janggu hourglass drum", "소리북": "sori-buk barrel drum",
    "박(拍)": "bak wooden clapper", "꽹과리": "kkwaenggwari small gong",
    "목어": "moktak wooden fish block", "프살테리": "plucked psaltery-style zither",
    "징": "jing large gong", "리코더": "wooden recorder flute",
}

PLACE_EN = {
    "PC": "Nolbu's reception room", "P1": "Nolbu's wife's room",
    "P2": "Heungbu's thatched hut", "P3": "the steward's room",
    "P4": "the neighboring laborer's house", "P5": "the herbalist's outdoor hut",
}
PLACE_UNIQUE_PRE = {"PC": "still nothing has happened here, a calm about to collapse"}
PLACE_UNIQUE_POST = {"PC": "where the body was laid, hushed stillness, low-register pressure"}

PERSONA_EN = {"가련": "pitiable", "표준": "composed, neutral", "음험": "sinister"}

CUE_EN = {
    "intro":         ("the original tale's narration ('once upon a time...')", "bright, fable-like, nostalgic"),
    "discovery":     ("the moment the body is found, the story's turn", "shock, sudden stillness, dissonance, a hard stop"),
    "investigation": ("exploring the scene and location", "calm tension, focused observation"),
    "interrogation": ("questioning a suspect", "a character's inner psychology, subtle pressure"),
    "climax":        ("the decisive clue is revealed, the final accusation", "mounting conviction and dread"),
    "reveal":        ("the true culprit is revealed, a confession", "tragic catharsis, the sorrow of the motive"),
    "ending":        ("the case is closed, epilogue", "resignation, a lingering feeling, a return to the original tale"),
}

EVENT_EN = {
    "round_start":     ("a new round begins, investigation resumes", "time closing in again"),
    "clue_found":      ("a general clue is revealed", "small thrill of discovery, a brief rise"),
    "clue_forensic":   ("a coroner's report or body-related clue is revealed", "the cold weight of fact"),
    "clue_decisive":   ("the decisive clue is revealed, the door to the truth", "everything clicking into place, conviction and a shiver"),
    "red_herring":     ("a misleading clue, red herring", "plausible but slightly off, a cadence resolved a half-step wrong"),
    "search_hit":      ("a search finds the right word", "the small satisfaction of finding the answer"),
    "search_miss":     ("a search returns nothing", "a false lead, two notes falling weakly"),
    "accusation":      ("the culprit-naming stage, final deduction", "the detective's moment when the pieces click together"),
    "verdict_correct": ("culprit, weapon, and motive are all correct", "a victory tinged with relief and bitterness"),
    "verdict_wrong":   ("the accusation fails", "crumbling certainty, the air cooling fast"),
    "life_lost":       ("one life or chance is used up", "one irreversible blow"),
    "time_low":        ("few interrogation turns remain", "a hurrying pulse"),
}

CHAR_WHEN_EN = {
    "C1": "questioning Heungbu", "C2": "questioning Nolbu's wife",
    "C3": "questioning the steward Maeho", "C4": "questioning the neighboring laborer",
    "C5": "questioning the herbalist old woman",
}


def build_tags(cat, cid, c):
    """json/ 의 bgm.*.prompt(한국어 섞인 원문) 대신, 구조화된 필드를 영어로
    번역해 ace_tags() 와 같은 짧은 태그 목록을 만든다. 위 헤더 주석 참고."""
    instruments = [INSTR_EN.get(k, k) for k in c.get("instruments", [])]
    bpm = c.get("bpm")
    dyn = c.get("dynamics")

    if cat == "cues":
        when_en, mood_en = CUE_EN[cid]
    elif cat == "event_cues":
        when_en, mood_en = EVENT_EN[cid]
    elif cat == "place_cues":
        if cid.endswith("_pre_murder"):
            pid, state = cid[: -len("_pre_murder")], "pre_murder"
        else:
            pid, state = cid[: -len("_post_murder")], "post_murder"
        place = PLACE_EN[pid]
        if state == "pre_murder":
            mood_en = PLACE_UNIQUE_PRE.get(pid, f"peaceful, everyday air of {place}, before anything has happened")
            when_en = f"exploring {place} (before the murder)"
        else:
            mood_en = PLACE_UNIQUE_POST.get(pid, "the same place turned cold, warmth drained, suspicion settling in")
            when_en = f"exploring {place} (after the murder)"
    elif cat == "character_themes":
        when_en = CHAR_WHEN_EN[cid]
        persona_en = PERSONA_EN.get(c.get("persona_flavor"), c.get("persona_flavor"))
        mood_en = f"{persona_en} character theme, layer added as pressure rises"
    else:
        raise ValueError(cat)

    tags = ["instrumental", "no vocals", "Korean traditional folk",
            "gyemyeonjo pentatonic mode", f"{bpm} BPM", dyn, mood_en, when_en]
    tags += instruments
    return tags


## 6. 생성

In [ ]:
STEPS      = 8 if IS_TURBO else 32     # turbo 8 / base 32~100
CFG_SCALE  = 7.0                       # turbo 에서는 무시된다
N_EXPANDED = 1  # 큐가 36개라 후보는 1개씩만 (필요하면 나중에 늘림)

for cat, cid, c in jobs:
    secs = parse_seconds(c.get("form"), default=60)
    outdir = OUT_DIR / PILOT_ID / cat
    outdir.mkdir(parents=True, exist_ok=True)
    skip_check = list(outdir.glob(f"{cid}_*"))
    if skip_check:
        print("   (이미 있음, 건너뜀)", cat, cid)
        continue

    params = GenerationParams(
        task_type="text2music",
        caption=", ".join(build_tags(cat, cid, c)),
        lyrics="[Instrumental]",
        instrumental=True,
        duration=float(secs),
        inference_steps=STEPS,
        guidance_scale=CFG_SCALE,
        seed=0,
        thinking=False,
    )
    config = GenerationConfig(
        batch_size=N_EXPANDED,
        use_random_seed=False,
        seeds=list(range(N_EXPANDED)),
        audio_format="wav",
    )
    res = generate_music(dit, llm, params, config, save_dir=str(outdir))
    if not res.success:
        print("FAIL", cat, cid, "->", res.error)
        continue
    for i, a in enumerate(res.audios):
        src_path = pathlib.Path(a.get("path", a) if isinstance(a, dict) else a)
        if src_path.exists() and src_path.name != f"{cid}_{i}{src_path.suffix}":
            src_path.rename(outdir / f"{cid}_{i}{src_path.suffix}")
    print("OK", cat, cid)

print("\n확장 BGM 저장 위치:", OUT_DIR / PILOT_ID)


## 7. 정리

In [ ]:
del dit, llm
free()